# DCGAN sinh ảnh khuôn mặt (CelebA 64x64) - huấn luyện trên Google Colab

**Chuẩn bị:** Runtime → Change runtime type → **GPU (T4)**. Trên Google Drive tạo thư mục `MyDrive/TTNT_NhomDHN/` chứa: thư mục `src/`, `requirements.txt` và file `archive.zip` (CelebA từ Kaggle).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = '/content/drive/MyDrive/TTNT_NhomDHN'   # thư mục dự án trên Drive
%cd /content
!rm -rf /content/work && mkdir -p /content/work
!cp -r {PROJ}/src {PROJ}/requirements.txt /content/work/
%cd /content/work
!nvidia-smi -L
!pip -q install -r requirements.txt

## Giai đoạn 1 - Tiền xử lý (center crop 148 → resize 64x64, lưu uint8)
Đọc thẳng từ `archive.zip`, ghi `celeba_64.npy` (~2.5 GB) lên đĩa local của Colab để đọc nhanh.

In [ ]:
!python -m src.preprocess --src {PROJ}/archive.zip --dst /content/data/celeba_64.npy

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load('/content/data/celeba_64.npy', mmap_mode='r')
print(d.shape, d.dtype)
fig, ax = plt.subplots(2, 8, figsize=(16, 4))
for a, i in zip(ax.flat, np.random.choice(len(d), 16, replace=False)):
    a.imshow(d[i]); a.axis('off')
plt.show()

## Giai đoạn 2 - Kiểm tra kiến trúc G / D

In [ ]:
!python -m src.models

## Giai đoạn 3 - Huấn luyện
Checkpoint, ảnh mẫu và log được lưu thẳng vào Drive nên không mất khi Colab ngắt kết nối; chạy lại cell này với `--resume` để tiếp tục.

In [ ]:
OUT = f'{PROJ}/outputs/run1'
!python -m src.train --data /content/data/celeba_64.npy --out {OUT} --epochs 25 --batch-size 128 --fid-every 5 --workers 2 --resume

In [ ]:
from IPython.display import Image, display
import glob
display(Image(f'{OUT}/training_curves.png'))
last = sorted(glob.glob(f'{OUT}/samples/epoch_*.png'))[-1]
print(last); display(Image(last))

## Giai đoạn 4 - Đánh giá (FID, đa dạng, nội suy, ảnh gần nhất)

In [ ]:
!python -m src.evaluate --ckpt {OUT}/best_fid.pt --data /content/data/celeba_64.npy --out {OUT}/eval --n-fid 10000 --save-images 1000

In [ ]:
for f in ['generated_grid', 'real_grid', 'interpolation', 'nearest_neighbors']:
    print(f); display(Image(f'{OUT}/eval/{f}.png'))